# 🌲 Notebook 02: Purchase Prediction – Random Forest

**Đồ án:** Xây dựng hệ thống E-commerce tích hợp Machine Learning

**Mục tiêu:** Dự đoán xem khách hàng có **quay lại mua hàng trong 30 ngày tới** hay không.

**Thuật toán:** Random Forest Classifier (Scikit-learn)

**Notebook này thực hiện:**
1. Load dữ liệu đã xử lý từ Notebook 01
2. Chuẩn bị data: scaling, train/test split
3. Baseline model (Logistic Regression) để so sánh
4. Train Random Forest với GridSearchCV
5. Đánh giá model: Accuracy, F1, ROC-AUC, Confusion Matrix
6. Phân tích Feature Importance
7. Cross-validation (K-Fold)
8. Export model `.joblib` để tích hợp vào FastAPI

---

> ⚠️ **Yêu cầu:** Chạy Notebook 01 trước để tạo file `customer_features.csv` trên Google Drive.

## 0. Setup & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings

from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Import thành công!')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ecommerce-ml'
DATA_DIR = f'{PROJECT_DIR}/data'
MODEL_DIR = f'{PROJECT_DIR}/models'

## 1. Load dữ liệu đã xử lý

In [ ]:
# Load customer features
customer_data = pd.read_csv(f'{DATA_DIR}/customer_features.csv')

# Load config
with open(f'{DATA_DIR}/config.json', 'r') as f:
    config = json.load(f)

FEATURE_COLUMNS = config['feature_columns']
TARGET = config['target']

print(f'✅ Loaded customer_features.csv')
print(f'   Shape: {customer_data.shape}')
print(f'   Features: {len(FEATURE_COLUMNS)}')
print(f'   Target: {TARGET}')
print(f'\n📊 Class distribution:')
print(customer_data[TARGET].value_counts())

customer_data.head()

## 2. Chuẩn bị dữ liệu cho Training

In [ ]:
# Tách X (features) và y (target)
X = customer_data[FEATURE_COLUMNS].copy()
y = customer_data[TARGET].copy()

# Xử lý giá trị NaN / Infinity
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nClass balance:')
print(f'  0 (Không mua): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)')
print(f'  1 (Sẽ mua):    {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)')

In [ ]:
# Train/Test Split (80/20, stratified để giữ tỷ lệ class)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Quan trọng: giữ tỷ lệ class giống nhau trong train/test
)

print(f'📊 Split Data:')
print(f'  Train: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'  Test:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\n  Train class balance: {y_train.mean()*100:.1f}% positive')
print(f'  Test class balance:  {y_test.mean()*100:.1f}% positive')

In [ ]:
# Feature Scaling (StandardScaler)
# Random Forest không yêu cầu scaling, nhưng cần cho Logistic Regression
# và để lưu scaler cho API endpoint sau này

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Chuyển về DataFrame để giữ tên cột
X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_COLUMNS, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=FEATURE_COLUMNS, index=X_test.index)

print('✅ Scaling hoàn tất!')
print(f'   Mean ≈ 0: {X_train_scaled.mean().mean():.6f}')
print(f'   Std ≈ 1:  {X_train_scaled.std().mean():.6f}')

## 3. Baseline Model: Logistic Regression

Train một model đơn giản trước để có baseline so sánh với Random Forest.

In [ ]:
# Logistic Regression (dùng scaled data)
lr = LogisticRegression(
    random_state=42,
    class_weight='balanced',  # Xử lý class imbalance
    max_iter=1000
)
lr.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('=' * 60)
print('📊 BASELINE: LOGISTIC REGRESSION')
print('=' * 60)
print(classification_report(y_test, y_pred_lr,
                             target_names=['Không mua (0)', 'Sẽ mua (1)']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}')

## 4. Random Forest – Training

### 4.1. Default Parameters

In [ ]:
# Random Forest với default parameters (sử dụng unscaled data)
# Random Forest KHÔNG cần scaling vì dùng decision trees

rf_default = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',  # Rất quan trọng cho imbalanced data
    random_state=42,
    n_jobs=-1  # Sử dụng tất cả CPU cores
)

print('🌲 Training Random Forest (default params)...')
rf_default.fit(X_train, y_train)

# Predictions
y_pred_rf_default = rf_default.predict(X_test)
y_prob_rf_default = rf_default.predict_proba(X_test)[:, 1]

print('\n' + '=' * 60)
print('📊 RANDOM FOREST (DEFAULT PARAMETERS)')
print('=' * 60)
print(classification_report(y_test, y_pred_rf_default,
                             target_names=['Không mua (0)', 'Sẽ mua (1)']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf_default):.4f}')

### 4.2. Hyperparameter Tuning với GridSearchCV

Tìm bộ hyperparameters tốt nhất bằng Cross-Validation.

In [ ]:
# GridSearchCV - tìm hyperparameters tối ưu
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced']
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# StratifiedKFold đảm bảo class balance trong mỗi fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('🔍 Bắt đầu GridSearchCV...')
print(f'   Tổng combinations: {np.prod([len(v) for v in param_grid.values()])}')
print(f'   × 5 folds = {np.prod([len(v) for v in param_grid.values()]) * 5} fits')
print('   ⏳ Có thể mất 5-15 phút...\n')

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',  # Optimize cho F1-Score
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print(f'\n✅ GridSearchCV hoàn tất!')
print(f'\n🏆 Best Parameters:')
for param, value in grid_search.best_params_.items():
    print(f'   {param}: {value}')
print(f'\n📊 Best CV F1-Score: {grid_search.best_score_:.4f}')

In [ ]:
# Lấy best model
best_rf = grid_search.best_estimator_

# Predictions với best model
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

print('=' * 60)
print('📊 RANDOM FOREST (BEST PARAMETERS - GridSearchCV)')
print('=' * 60)
print(classification_report(y_test, y_pred_best,
                             target_names=['Không mua (0)', 'Sẽ mua (1)']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob_best):.4f}')

## 5. Đánh giá Model chi tiết

### 5.1. So sánh các model

In [ ]:
# So sánh tất cả models
models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'RF (Default)': (y_pred_rf_default, y_prob_rf_default),
    'RF (Tuned)': (y_pred_best, y_prob_best),
}

comparison = []
for name, (y_pred, y_prob) in models.items():
    comparison.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
    })

comparison_df = pd.DataFrame(comparison).set_index('Model')

print('=' * 80)
print('🏆 SO SÁNH CÁC MODEL')
print('=' * 80)
print(comparison_df.round(4).to_string())

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
comparison_df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='white')
ax.set_title('So sánh hiệu suất các Model', fontsize=16, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Target F1 = 0.7')
plt.tight_layout()
plt.show()

### 5.2. Confusion Matrix

In [ ]:
# Confusion Matrix cho best model
cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Không mua (0)', 'Sẽ mua (1)'],
            yticklabels=['Không mua (0)', 'Sẽ mua (1)'])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Thực tế (Actual)')
axes[0].set_xlabel('Dự đoán (Predicted)')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Không mua (0)', 'Sẽ mua (1)'],
            yticklabels=['Không mua (0)', 'Sẽ mua (1)'])
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Thực tế (Actual)')
axes[1].set_xlabel('Dự đoán (Predicted)')

plt.suptitle('🎯 Random Forest (Tuned) - Confusion Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# In giải thích
tn, fp, fn, tp = cm.ravel()
print(f'\n📊 Chi tiết:')
print(f'  True Negatives (TN):  {tn:,} - Dự đoán KHÔNG mua, đúng là không mua ✅')
print(f'  False Positives (FP): {fp:,} - Dự đoán SẼ MUA, nhưng thực tế không mua ❌')
print(f'  False Negatives (FN): {fn:,} - Dự đoán KHÔNG mua, nhưng thực tế có mua ❌')
print(f'  True Positives (TP):  {tp:,} - Dự đoán SẼ MUA, đúng là có mua ✅')

### 5.3. ROC Curve & Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- ROC Curve ---
for name, (_, y_prob) in models.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC=0.500)')
axes[0].set_title('📈 ROC Curve', fontsize=14, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# --- Precision-Recall Curve ---
for name, (_, y_prob) in models.items():
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(recall, precision, linewidth=2, label=f'{name} (AP={ap:.3f})')

axes[1].set_title('📈 Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

### 5.4. Feature Importance

In [ ]:
# Feature Importance từ Random Forest
feature_imp = pd.Series(
    best_rf.feature_importances_,
    index=FEATURE_COLUMNS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feature_imp)))
feature_imp.plot(kind='barh', ax=ax, color=colors)
ax.set_title('🏅 Feature Importance (Random Forest)', fontsize=16, fontweight='bold')
ax.set_xlabel('Importance Score')

# Thêm giá trị trên bar
for i, (val, name) in enumerate(zip(feature_imp.values, feature_imp.index)):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\n🏅 Feature Importance Ranking:')
for rank, (feat, imp) in enumerate(feature_imp.sort_values(ascending=False).items(), 1):
    bar = '█' * int(imp * 100)
    print(f'  {rank:2d}. {feat:30s} {imp:.4f} {bar}')

## 6. Cross-Validation (K-Fold)

Đánh giá model trên nhiều fold để đảm bảo kết quả ổn định, không bị overfitting.

In [ ]:
# 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = ['accuracy', 'f1', 'roc_auc', 'precision', 'recall']
cv_results = {}

print('🔄 Running 5-Fold Cross-Validation...\n')

for metric in scoring_metrics:
    scores = cross_val_score(best_rf, X, y, cv=cv, scoring=metric, n_jobs=-1)
    cv_results[metric] = scores
    print(f'  {metric:12s}: {scores.mean():.4f} ± {scores.std():.4f}  '
          f'(min={scores.min():.4f}, max={scores.max():.4f})')

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
cv_df = pd.DataFrame(cv_results)
bp = ax.boxplot([cv_df[col] for col in cv_df.columns],
                labels=cv_df.columns,
                patch_artist=True,
                medianprops=dict(color='red', linewidth=2))

colors_box = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('📊 5-Fold Cross-Validation Results', fontsize=16, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5)
ax.text(0.5, 0.71, 'Target = 0.7', fontsize=10, alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Phân tích Probability Distribution

Xem phân bố xác suất dự đoán để chọn threshold cho promotion popup.

In [ ]:
# Phân bố xác suất dự đoán
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
for label, color, name in [(0, '#e74c3c', 'Không mua'), (1, '#2ecc71', 'Sẽ mua')]:
    mask = y_test == label
    axes[0].hist(y_prob_best[mask], bins=50, alpha=0.6, label=name, color=color, density=True)

axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold=0.5')
axes[0].axvline(x=0.7, color='orange', linestyle='--', label='Promotion threshold=0.7')
axes[0].set_title('Phân bố xác suất dự đoán', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Probability of Purchase')
axes[0].set_ylabel('Density')
axes[0].legend()

# Threshold analysis
thresholds = np.arange(0.3, 0.9, 0.05)
f1_scores = []
precisions = []
recalls = []

for thresh in thresholds:
    y_pred_thresh = (y_prob_best >= thresh).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_thresh))
    precisions.append(precision_score(y_test, y_pred_thresh, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_thresh))

axes[1].plot(thresholds, f1_scores, 'b-o', label='F1-Score', linewidth=2)
axes[1].plot(thresholds, precisions, 'g--', label='Precision', linewidth=1.5)
axes[1].plot(thresholds, recalls, 'r--', label='Recall', linewidth=1.5)

best_thresh_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_thresh_idx]
axes[1].axvline(x=best_thresh, color='orange', linestyle=':', linewidth=2,
                 label=f'Best threshold={best_thresh:.2f}')

axes[1].set_title('F1 / Precision / Recall vs Threshold', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\n🎯 Best threshold for F1: {best_thresh:.2f} (F1={max(f1_scores):.4f})')

## 8. Export Model

Lưu tất cả artifacts cần thiết ra Google Drive để dùng trong:
- **Notebook 03, 04**: K-Means, KNN (Sprint 2)
- **FastAPI Backend**: Load model và serve predictions (Sprint 3)

In [ ]:
import os

# Đảm bảo thư mục tồn tại
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. Save Random Forest model
rf_path = f'{MODEL_DIR}/random_forest_model.joblib'
joblib.dump(best_rf, rf_path)
print(f'✅ Random Forest model → {rf_path}')
print(f'   File size: {os.path.getsize(rf_path) / 1024 / 1024:.1f} MB')

# 2. Save StandardScaler
scaler_path = f'{MODEL_DIR}/scaler_rfm.joblib'
joblib.dump(scaler, scaler_path)
print(f'✅ StandardScaler → {scaler_path}')

# 3. Save model metadata
model_metadata = {
    'model_type': 'RandomForestClassifier',
    'best_params': grid_search.best_params_,
    'feature_columns': FEATURE_COLUMNS,
    'target': TARGET,
    'metrics': {
        'accuracy': accuracy_score(y_test, y_pred_best),
        'precision': precision_score(y_test, y_pred_best),
        'recall': recall_score(y_test, y_pred_best),
        'f1_score': f1_score(y_test, y_pred_best),
        'roc_auc': roc_auc_score(y_test, y_prob_best),
    },
    'cv_f1_mean': float(cv_results['f1'].mean()),
    'cv_f1_std': float(cv_results['f1'].std()),
    'best_threshold': float(best_thresh),
    'promotion_threshold': 0.7,
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'n_features': len(FEATURE_COLUMNS),
}

metadata_path = f'{MODEL_DIR}/rf_model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2, default=str)
print(f'✅ Model metadata → {metadata_path}')

print('\n' + '=' * 60)
print('📦 CÁC FILE ĐÃ EXPORT:')
print('=' * 60)
for f in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, f)
    size = os.path.getsize(fpath)
    if size > 1024 * 1024:
        print(f'  📄 {f} ({size / 1024 / 1024:.1f} MB)')
    else:
        print(f'  📄 {f} ({size / 1024:.1f} KB)')

In [ ]:
# Tùy chọn: Download files về máy local
# Uncomment nếu muốn download trực tiếp từ Colab

# from google.colab import files
# files.download(rf_path)
# files.download(scaler_path)
# files.download(metadata_path)
# files.download(f'{MODEL_DIR}/label_encoder_country.joblib')

## 9. Quick Test: Mô phỏng prediction cho 1 khách hàng

Thử predict cho 1 khách hàng cụ thể để kiểm tra model hoạt động đúng.

In [ ]:
# Load lại model (mô phỏng như FastAPI sẽ làm)
loaded_model = joblib.load(rf_path)
loaded_scaler = joblib.load(scaler_path)

# Lấy 1 khách hàng mẫu từ test set
sample_idx = X_test.index[0]
sample_features = X_test.loc[[sample_idx]]
actual_label = y_test.loc[sample_idx]

# Predict
prediction = loaded_model.predict(sample_features)[0]
probability = loaded_model.predict_proba(sample_features)[0]

customer_id = customer_data.loc[sample_idx, 'Customer ID']

print('=' * 60)
print(f'🧪 TEST PREDICTION CHO KHÁCH HÀNG #{int(customer_id)}')
print('=' * 60)
print(f'\n📋 Features:')
for feat in FEATURE_COLUMNS:
    print(f'   {feat:30s}: {sample_features[feat].values[0]:.4f}')

print(f'\n🎯 Kết quả:')
print(f'   Prediction: {"SẼ MUA LẠI ✅" if prediction == 1 else "KHÔNG MUA LẠI ❌"}')
print(f'   Probability: {probability[1]:.4f} ({probability[1]*100:.1f}%)')
print(f'   Actual:      {"Đã mua lại ✅" if actual_label == 1 else "Không mua lại ❌"}')
print(f'   Đúng/Sai:    {"✅ ĐÚNG" if prediction == actual_label else "❌ SAI"}')

# Promotion check
PROMOTION_THRESHOLD = 0.7
if probability[1] >= PROMOTION_THRESHOLD:
    print(f'\n   🎁 → HIỂN THỊ KHUYẾN MÃI (prob {probability[1]:.2f} ≥ {PROMOTION_THRESHOLD})')
else:
    print(f'\n   ⏸️ → Không hiển thị khuyến mãi (prob {probability[1]:.2f} < {PROMOTION_THRESHOLD})')

---

## ✅ Tổng kết Notebook 02

| Mục | Kết quả |
|-----|--------|
| Model | Random Forest Classifier |
| Tuning | GridSearchCV (5-Fold CV) |
| Features | 13 features (RFM + Behavioral) |
| Export | `.joblib` files trên Google Drive |

### Tiếp theo:
- **Notebook 03**: Customer Segmentation (K-Means + PCA)
- **Notebook 04**: Product Recommendation (KNN)
- Sau đó retrain Random Forest với thêm `segment_id` từ K-Means

### Files đã export:
```
ecommerce-ml/models/
├── random_forest_model.joblib
├── scaler_rfm.joblib
├── label_encoder_country.joblib
└── rf_model_metadata.json
```